# Olist: Do Late Deliveries Lower Customer Review Scores?

**Business question:** Does delivering after the estimated date significantly reduce customer satisfaction, and does the effect survive controlling for customer state and product category?

**Hypotheses**
- H0: Review scores are the same for late and on-time deliveries.
- H1: Late deliveries receive lower review scores.

**Data:** BigQuery `myprojctexample.Olist_analysis` (cleaned_orders, cleaned_reviews, cleaned_customers, cleaned_order_items, cleaned_products).
**Unit of analysis:** one delivered order with one review (latest review per order).
**Definition:** an order is *late* if the delivery date is after `order_estimated_delivery_date`.

In [ ]:
import os

import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.formula.api as smf
from google.cloud import bigquery

# Configuration comes from environment variables, so no key path is stored in this notebook.
PROJECT = os.environ.get("GCP_PROJECT_ID", "myprojctexample")
DATASET = os.environ.get("BQ_DATASET", "Olist_analysis")
key_path = os.environ.get("BQ_KEY_PATH")

if key_path:
    client = bigquery.Client.from_service_account_json(key_path, project=PROJECT)
else:
    client = bigquery.Client(project=PROJECT)  # falls back to Application Default Credentials

def fmt_p(p):
    return "< 0.001" if p < 0.001 else f"= {p:.3f}"

## 1. Build the analysis table
Cleaning decisions: keep only delivered orders with a delivery timestamp, cast `review_score` from STRING to INT, keep one review per order (the latest), and take the category of the first item in the order.

In [ ]:
sql = f"""
WITH r AS (
  SELECT order_id, SAFE_CAST(review_score AS INT64) AS review_score
  FROM `{PROJECT}.{DATASET}.cleaned_reviews`
  QUALIFY ROW_NUMBER() OVER (
    PARTITION BY order_id ORDER BY review_creation_date DESC, review_id) = 1
),
cat AS (
  SELECT i.order_id, p.product_category_name AS category
  FROM `{PROJECT}.{DATASET}.cleaned_order_items` i
  JOIN `{PROJECT}.{DATASET}.cleaned_products` p USING (product_id)
  WHERE i.order_item_id = 1
)
SELECT
  o.order_id,
  r.review_score,
  c.customer_state,
  cat.category,
  DATE_DIFF(DATE(o.order_delivered_customer_date),
            o.order_estimated_delivery_date, DAY) AS days_vs_estimate
FROM `{PROJECT}.{DATASET}.cleaned_orders` o
JOIN r USING (order_id)
JOIN `{PROJECT}.{DATASET}.cleaned_customers` c USING (customer_id)
LEFT JOIN cat USING (order_id)
WHERE o.order_status = 'delivered'
  AND o.order_delivered_customer_date IS NOT NULL
  AND r.review_score IS NOT NULL
"""
df = client.query(sql).to_dataframe(create_bqstorage_client=False)
df["category"] = df["category"].fillna("unknown")
df = df.dropna(subset=["days_vs_estimate", "review_score"])
df["review_score"] = df["review_score"].astype(int)
df["days_vs_estimate"] = df["days_vs_estimate"].astype(int)
df["is_late"] = df["days_vs_estimate"] > 0
print(df.shape)
df.head()

## 2. Descriptive comparison

In [ ]:
summary = (df.groupby("is_late")["review_score"]
             .agg(n="count", mean="mean", median="median")
             .round(3))
summary["pct_1_2_star"] = df.groupby("is_late")["review_score"].apply(lambda s: (s <= 2).mean() * 100).round(1)
summary["pct_5_star"] = df.groupby("is_late")["review_score"].apply(lambda s: (s == 5).mean() * 100).round(1)
summary

## 3. Hypothesis tests
Review scores are ordinal (1-5) and heavily skewed, so **Mann-Whitney U** is the primary test. Welch's t-test, Cohen's d, and a chi-square on the 1-2 star share are secondary checks. With ~96K orders, p-values will be tiny, so **effect size matters more than p**.

In [ ]:
late = df.loc[df.is_late, "review_score"]
ontime = df.loc[~df.is_late, "review_score"]

# Primary: Mann-Whitney U (late scores stochastically lower)
u, p_mw = stats.mannwhitneyu(late, ontime, alternative="less")
rank_biserial = 1 - 2 * u / (len(late) * len(ontime))

# Secondary: Welch t-test + Cohen's d
t, p_t = stats.ttest_ind(late, ontime, equal_var=False, alternative="less")
pooled = np.sqrt(((len(late)-1)*late.var(ddof=1) + (len(ontime)-1)*ontime.var(ddof=1))
                 / (len(late)+len(ontime)-2))
d = (late.mean() - ontime.mean()) / pooled

# 95% CI for the mean gap (on-time minus late)
gap = ontime.mean() - late.mean()
se = np.sqrt(ontime.var(ddof=1)/len(ontime) + late.var(ddof=1)/len(late))

# Share of 1-2 star reviews
ct = pd.crosstab(df.is_late, df.review_score <= 2)
chi2, p_chi, _, _ = stats.chi2_contingency(ct)

print(f"Mann-Whitney U: p {fmt_p(p_mw)} | rank-biserial effect size: {rank_biserial:.3f}")
print(f"Welch t = {t:.1f}, p {fmt_p(p_t)} | Cohen's d = {d:.2f}")
print(f"Mean gap (on-time - late): {gap:.3f} stars, 95% CI [{gap-1.96*se:.3f}, {gap+1.96*se:.3f}]")
print(f"Chi-square on 1-2 star share: {chi2:.0f}, p {fmt_p(p_chi)}")

## 4. Dose-response: does the rating fall as delays get longer?
If lateness were not driving dissatisfaction, we would not expect a clean step-down pattern as days late increase.

In [ ]:
bins = [-np.inf, -8, -1, 0, 3, 7, 14, np.inf]
labels = ["8+ days early", "1-7 days early", "on estimate day",
          "1-3 days late", "4-7 days late", "8-14 days late", "15+ days late"]
df["bucket"] = pd.cut(df.days_vs_estimate, bins=bins, labels=labels, right=True)
dose = (df.groupby("bucket", observed=True)["review_score"]
          .agg(n="count", mean_score="mean")
          .round(3))
dose["pct_1_2_star"] = df.groupby("bucket", observed=True)["review_score"].apply(lambda s: (s <= 2).mean()*100).round(1)
dose

## 5. Confounder checks
Late orders could differ from on-time orders in *where* they ship and *what* they contain. Two checks:

1. **Stratified comparison** of late vs on-time within each customer state and product category.
2. **Regression with fixed effects** for state and category, estimating the late-delivery penalty after controlling for both.

In [ ]:
def stratified_gap(data, col, min_late=30):
    g = data.groupby([col, "is_late"])["review_score"].agg(["mean", "count"]).unstack("is_late")
    g.columns = ["mean_ontime", "mean_late", "n_ontime", "n_late"]
    g = g[g.n_late >= min_late].copy()
    g["gap"] = g.mean_ontime - g.mean_late
    w = g.n_late / g.n_late.sum()
    return g, (w * g.gap).sum()

for col in ["customer_state", "category"]:
    g, wgap = stratified_gap(df, col)
    print(f"{col}: {len(g)} strata | gap > 0 in {(g.gap > 0).sum()}/{len(g)} | late-weighted gap = {wgap:.3f} stars")

In [ ]:
model = smf.ols("review_score ~ is_late + C(customer_state) + C(category)",
                data=df.assign(is_late=df.is_late.astype(int))).fit(cov_type="HC1")
coef = model.params["is_late"]
lo, hi = model.conf_int().loc["is_late"]
print(f"Late-delivery effect controlling for state and category: {coef:.3f} stars (95% CI [{lo:.3f}, {hi:.3f}])")

## 6. Results (run on the full dataset)

| Metric | On-time | Late |
|---|---|---|
| Orders | 89,443 | 6,381 (6.7%) |
| Mean review score | 4.29 | 2.27 |
| Median | 5 | 1 |
| 1-2 star reviews | 9.3% | 62.4% |
| 5-star reviews | 62.3% | 16.6% |

- **Mann-Whitney U:** p < 0.001; rank-biserial effect size = 0.64
- **Cohen's d:** -1.71 (very large)
- **Mean gap:** 2.02 stars, 95% CI [1.98, 2.06]
- **Dose-response:** mean score falls from 4.32 (8+ days early) to 3.29 (1-3 days late), 2.11 (4-7 days late), and about 1.7 (8+ days late). The 1-2 star share rises from 9% to roughly 80%.
- **Confounders:** the gap is positive in all 21 states and all 29 product categories with 30+ late orders. The late-weighted gap is about 2.0 stars within states and 2.02 within categories, essentially unchanged from the raw 2.02. A regression with state and category fixed effects estimates the late-delivery effect at -1.99 stars (95% CI -2.03 to -1.95).

## 7. Conclusion and limitations
We reject H0: late delivery is strongly associated with lower review scores, and the association holds within states and product categories, with a clear dose-response pattern.

**Limitations**
- This is observational data, so the results show a strong association rather than proven causation. Unmeasured factors (seller quality, delivery distance, product condition) could contribute.
- Only customers who left a review are included; non-reviewers may differ.
- Review score is ordinal, so the mean gap is a summary; the 1-2 star share is the more interpretable business metric.
- Seller-level or distance-based controls would be a natural next step.

**Business takeaway:** about 6.7% of orders arrive late, yet they account for a disproportionate share of 1-2 star reviews, so delivery-time reliability is a high-leverage satisfaction lever.